# Capítulo 17: Clustering

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto. É o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem daqui —
# no livro isso vem do `execute-dir: project` do Quarto.
import os
import sys

_raiz = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_raiz, "_quarto.yml")):
    _pai = os.path.dirname(_raiz)
    if _pai == _raiz:
        raise RuntimeError("raiz do projeto não encontrada (procurando _quarto.yml)")
    _raiz = _pai
os.chdir(_raiz)
if _raiz not in sys.path:
    sys.path.insert(0, _raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 20 de Grus (2019).

> Onde tínhamos tais cachos,
> que nos faziam nobremente loucos, não dementes.
>
> — Robert Herrick. No original, os cachos são *clusters* — a mesma palavra que dá nome ao capítulo, e um trocadilho que não sobrevive à tradução.

Todo modelo dos dezesseis capítulos anteriores tinha uma resposta certa em algum lugar. A flor era *setosa* ou *virginica*, o e-mail era spam ou não era, o usuário passava tantos minutos por dia no site, o dígito era um 7. É por isso que "funcionou" sempre pôde significar "acertou": havia um gabarito.

Este capítulo não tem gabarito. Os dados chegam sem rótulo nenhum, e a tarefa é descobrir que grupos existem neles. **É o único capítulo do livro inteiramente dedicado a aprendizado não supervisionado** — a [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) já tinha usado o PCA, que também dispensa rótulos, mas ali ele era um preparo para modelos supervisionados, e não a resposta final. A mudança é maior do que parece à primeira vista: sem rótulo não há acerto, sem acerto não há acurácia, precisão, revocação nem R², e sem resposta certa não faz sentido reservar um conjunto de teste. Todo o vocabulário que o [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) construiu para julgar um modelo deixa de se aplicar de uma vez só.

A pergunta que sobra é mais desconfortável e mais parecida com o trabalho real: *este agrupamento é útil?* A resposta depende do que você vai fazer com ele, e não existe um número que a dê. A [seção 17.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/04-escolhendo-k.html) é onde isso dói — não há um *k* correto esperando ser descoberto, e a métrica óbvia para escolhê-lo tem exatamente o mesmo defeito do R² da [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html): ela sempre melhora, inclusive na resposta mais inútil possível.

O capítulo constrói dois algoritmos. O **k-means** é o algoritmo de agrupamento mais usado que existe, alterna dois passos exatos até parar de mudar, e — como o [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) já tinha avisado — não passa por gradiente nenhum. O **clustering hierárquico bottom-up** vai pelo caminho oposto: começa com um grupo por ponto e funde os mais próximos até sobrar um só, produzindo uma árvore da qual se extrai qualquer número de grupos depois. Os dois discordam sobre os mesmos vinte pontos, e essa discordância é o conteúdo, não um defeito.

Ao final deste capítulo, você será capaz de:

- Distinguir aprendizado supervisionado de não supervisionado, e explicar quais métricas deixam de fazer sentido quando não há rótulo
- Implementar o k-means do zero e argumentar por que ele sempre converge, mas para um ótimo local que depende da inicialização
- Usar a curva do erro por *k* para escolher um número de grupos, e reconhecer por que essa métrica, sozinha, não pode decidir
- Aplicar clustering a um problema concreto de compressão — reduzir uma imagem a cinco cores — e diagnosticar o que ele sacrifica no caminho
- Implementar clustering hierárquico bottom-up e mostrar como a troca de um único argumento muda o resultado por completo
- Julgar um agrupamento sem gabarito: dizer o que "funcionou" significa quando não existe resposta certa

## Seções

| Seção | Tópico |
|---|---|
| [17.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/01-a-ideia.html) | A Ideia |
| [17.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/02-o-modelo.html) | O Modelo |
| [17.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/03-exemplo-encontros.html) | Exemplo: Encontros |
| [17.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/04-escolhendo-k.html) | Escolhendo k |
| [17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html) | Exemplo: Clustering de Cores |
| [17.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/06-clustering-hierarquico.html) | Clustering Hierárquico Bottom-Up |

## A Ideia

> **📌 Nota**
>
> Esta seção corresponde a *The Idea*, do capítulo 20 de Grus (2019).

Um botânico mediu as pétalas de uma flor e escreveu *virginica* ao lado. Um usuário abriu a caixa de entrada e apertou "marcar como spam". Um relógio contou os minutos que alguém passou no site. Uma pessoa olhou um rabisco de 28 por 28 pixels e digitou 7.

Antes de qualquer modelo deste livro rodar, alguém já tinha respondido à pergunta que ele foi treinado para responder — e o que o modelo aprendeu, em todos os casos, foi a imitar essa pessoa. O rótulo nunca foi uma propriedade do dado: é trabalho humano, feito antes, e dele saía tudo o que vinha depois, do treino à métrica.

Este é o único capítulo do livro cujo objetivo é **agrupar**, não prever — e o único em que ninguém respondeu nada antes. (A [seção 7.8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/08-reducao-de-dimensionalidade.html) também dispensa rótulos, mas ali a tarefa era resumir as dimensões, não descobrir grupos.)

> **❗ Importante — O que sai de cena**
>
> O [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) construiu o vocabulário com que este livro julga um modelo: dividir treino e teste, medir acurácia, precisão e revocação, desconfiar de um ajuste bom demais. Todo esse vocabulário depende de uma coisa só — a existência de uma resposta correta contra a qual comparar a previsão.
>
> Aqui não há uma. Não há rótulo, portanto não há acerto; não há acerto, portanto não há acurácia, matriz de confusão nem R². E não há conjunto de teste, porque "generalizar para dados que o modelo não viu" pressupõe que exista algo a acertar neles.
>
> A pergunta muda de forma. Ela deixa de ser *o modelo acerta?* e passa a ser **o agrupamento é útil?** — e a resposta a essa depende inteiramente do que você vai fazer com o agrupamento depois.

Modelos que aprendem a partir de dados rotulados se chamam **supervisionados**; a [seção 8.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/02-o-que-e-machine-learning.html) já usou o termo. Clustering é o exemplo mais completo deste livro do outro tipo: **não supervisionado**, em que se trabalha com dados completamente sem rótulo — ou em que os rótulos existem e nós os ignoramos de propósito.

### Os grupos estão no dado

A observação que sustenta o capítulo inteiro é simples: sempre que você olha para uma fonte de dados, é provável que ela forme **grupos** por conta própria.

Um cadastro com o endereço de pessoas muito ricas de uma cidade não espalha os pontos uniformemente pelo mapa — há concentrações em alguns bairros e quase ninguém em outros. Um conjunto com quantas horas por semana cada pessoa trabalha tem um amontoado enorme em torno de 40. Um cadastro de eleitores com idade, renda e composição familiar forma agrupamentos que consultores políticos reconhecem e batizam: "mães de família da periferia", "aposentados sem filhos em casa", "jovens sem emprego formal".

Nenhum desses grupos foi colocado ali por um algoritmo. Eles são propriedade do fenômeno que gerou os dados. O que um algoritmo de clustering faz é **encontrá-los**.

Vale ver isso acontecendo com números. Suponha que uma pesquisa de emprego tenha coletado a jornada semanal de mil pessoas, num país onde a lei obriga o empregador a conceder certos benefícios a quem trabalha 20 horas por semana ou mais:

In [ ]:
# Figura: Jornada semanal de mil pessoas. Os dois grupos não foram desenhados: eles são o que sobra depois que a lei e o costume agiram sobre as jornadas.
import random
from matplotlib import pyplot as plt

random.seed(17)

# a maioria trabalha em tempo integral, perto de 40 horas
horas = [random.gauss(40, 3) for _ in range(800)]
# e uma minoria trabalha logo abaixo do limite que gera direito a benefícios:
# o empregador oferece jornadas mais curtas e nunca cruza as 20 horas.
# O `min` é o teto: quem cairia acima dele é reagendado para 19,9 -- e o
# pequeno acúmulo exatamente ali, na figura, É o teto aparecendo nos dados.
horas += [min(19.9, random.gauss(18.5, 1.0)) for _ in range(200)]

plt.hist(horas, bins=40, color="#4477aa", edgecolor="white")
plt.xlabel("horas trabalhadas por semana")
plt.ylabel("nº de pessoas")
plt.title("Ninguém disse ao histograma onde estão os grupos")
plt.show()

Os dados acima são inventados, mas o formato não é: o segundo amontoado, logo *abaixo* das 20 horas, é o que se espera de um empregador que evita cruzar o limite legal. Repare que o gráfico não recebeu nenhuma informação sobre grupos. Ele recebeu mil números e desenhou onde eles caem — e os dois montes apareceram sozinhos.

### Não existe o agrupamento correto

Em uma dimensão, como no histograma acima, o olho resolve. Em duas, também. O problema começa em dez, ou em três mil, e é aí que se precisa de um algoritmo. Mas o algoritmo herda uma dificuldade que já estava no problema, e não vai embora:

> **🔷 Conceito**
>
> **Em geral, não existe um agrupamento "correto".**
>
> O esquema que separa "jovens sem emprego formal" de "estudantes de pós-graduação" e o esquema que os junta num grupo só não são um certo e o outro errado. Cada um é mais adequado *segundo a própria métrica* de quão bons são os grupos.
>
> E "quão bons são os grupos" é uma escolha de quem modela, não um fato do conjunto de dados. Nos dezesseis capítulos anteriores, a métrica de sucesso vinha de fora, junto com os rótulos. Aqui ela vem de você.

Há uma segunda consequência, mais prática e igualmente inconveniente: **os grupos não se rotulam sozinhos**. O algoritmo devolve "grupo 0", "grupo 1", "grupo 2". Quem descobre que o grupo 1 é "quem compra só na promoção" é uma pessoa, olhando os dados que caíram dentro dele. Essa etapa não é opcional e não é automatizável — é a diferença entre uma partição e uma descoberta.

> **⚠️ Atenção — Dois `k` que não têm nada a ver um com o outro**
>
> O algoritmo desta seção se chama **k-means**, e o do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html) se chamava **k-vizinhos mais próximos**. Os dois têm um `k` no nome, usam distância euclidiana e param por aí.
>
> No k-vizinhos, `k` é *quantos vizinhos votam* na classificação de um ponto novo, e o modelo é supervisionado. No k-means, `k` é *quantos grupos existem*, e não há nada a classificar porque não há classe. Confundir os dois é um erro comum e caro; a semelhança está no nome, não na ideia.

> **💡 Dica — Na prática: o que se faz com isso**
>
> Sem rótulo e sem métrica externa, é justo perguntar para que serve. Os usos mais comuns:
>
> - **Segmentação.** Uma empresa com milhões de clientes e nenhuma etiqueta neles agrupa por comportamento de compra e trata cada grupo de um jeito. O critério de sucesso não é interno ao algoritmo: é se a campanha desenhada para o grupo 2 vendeu mais que a campanha genérica.
> - **Compressão e quantização.** Substituir cada ponto pelo centro do grupo dele reduz drasticamente o número de valores distintos. É exatamente o que a [seção 17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html) vai fazer com as cores de uma imagem.
> - **Exploração.** Antes de ter hipótese nenhuma, agrupar e olhar o que caiu junto é uma forma barata de descobrir que os dados têm estrutura — ou de descobrir que o "grupo" mais nítido é, na verdade, um defeito de coleta.
>
> O `scikit-learn` reúne isso em `sklearn.cluster`, com uma dúzia de algoritmos diferentes. Vale saber desde já que a lista é longa **porque** não existe resposta certa: cada algoritmo carrega uma definição diferente de "grupo" — bolas compactas, regiões densas, cadeias conectadas — e devolve respostas diferentes sobre os mesmos dados. As seções seguintes constroem dois deles e mostram, lado a lado, o quanto discordam.

## O Modelo

> **📌 Nota**
>
> Esta seção corresponde a *The Model*, do capítulo 20 de Grus (2019).

Cada entrada será, como de costume neste livro, um vetor num espaço de *d* dimensões — uma lista de números. O objetivo é duplo: identificar grupos de entradas parecidas e, quando fizer sentido, achar um **valor representativo** de cada grupo.

Os dois objetivos aparecem juntos com frequência. Se cada entrada é um vetor numérico que representa o título de um post de blog, os grupos podem revelar sobre o que os usuários escrevem — e o valor representativo de cada grupo é um título fictício que resume o assunto. Se cada entrada é uma cor de uma imagem com milhares delas, e você precisa imprimir uma versão com dez cores, os grupos dizem *quais* dez cores usar e o representante de cada grupo *é* a cor a ser impressa.

### k-means

Um dos métodos mais simples é o **k-means**. O número de grupos, *k*, é escolhido de antemão — não é descoberto, é decidido —, e a meta é dividir as entradas em conjuntos $S_1, \dots, S_k$ de modo a minimizar a soma total das distâncias ao quadrado de cada ponto até a média do grupo a que ele pertence.

Enunciada assim, a tarefa parece um problema de busca: basta experimentar todas as divisões possíveis e ficar com a melhor. O tamanho dessa busca é o primeiro obstáculo:

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def particoes(n: int, k: int) -> int:
    """De quantas formas se pode dividir n pontos em exatamente k grupos não vazios."""
    if k > n:
        return 0
    if k == 1 or k == n:
        return 1
    # o n-ésimo ponto ou entra num dos k grupos já formados pelos outros n-1...
    # ...ou abre sozinho o k-ésimo grupo
    return k * particoes(n - 1, k) + particoes(n - 1, k - 1)

def br(n: int) -> str:
    """separador de milhar no padrão brasileiro"""
    return f"{n:,}".replace(",", ".")

print(f"20 pontos em 3 grupos: {br(particoes(20, 3))}")
print(f"20 pontos em 5 grupos: {br(particoes(20, 5))}")
print(f"50 pontos em 5 grupos: {br(particoes(50, 5))}")

Vinte pontos. Vinte é um conjunto de dados que cabe numa folha de papel, e já são mais de meio bilhão de maneiras de dividi-lo em três. Encontrar a divisão ótima é, de fato, um problema difícil — e não do tipo que espera por um computador mais rápido.

> **🔷 Conceito**
>
> Por isso o k-means não procura o ótimo. Ele é um algoritmo **iterativo** que costuma achar um bom agrupamento, e consiste em repetir dois passos exatos (2 e 4) até nada mudar:
>
> 1. Comece com *k* médias, que são pontos no espaço de *d* dimensões.
> 2. **Atribua** cada ponto à média de que ele está mais próximo.
> 3. Se nenhuma atribuição mudou, pare: os grupos estão prontos.
> 4. Se alguma mudou, **recalcule** as médias e volte ao passo 2.
>
> Nenhum dos dois passos tem parâmetro a ajustar, taxa de aprendizado ou derivada. Cada um é uma conta fechada.
>
> O passo 1 tem uma dobra que o código adiante vai revelar: em vez de inventar coordenadas para *k* médias, ele sorteia uma atribuição aleatória para cada ponto e deixa o passo 4 produzir as médias iniciais. É a mesma receita começada meia volta antes — e é de onde vem toda a aleatoriedade do algoritmo.

> **📌 Nota — Otimização sem gradiente**
>
> O [Capítulo 5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/index.html) construiu o gradiente descendente e avisou que quatro modelos deste livro não passariam por lá. O k-means é o último deles, e é o mais claro dos quatro: existe uma quantidade a minimizar — a soma das distâncias ao quadrado —, e mesmo assim não se calcula derivada nenhuma.
>
> O motivo é que a variável sobre a qual se está otimizando não é contínua. "A que grupo o ponto 7 pertence" é uma escolha entre *k* alternativas discretas; não há um passo pequeno nessa direção, então não há gradiente a descer. O que existe são dois passos que, cada um à sua maneira, não pioram a resposta.

### O código

Vamos precisar de uma função auxiliar que conte em quantas coordenadas dois vetores diferem. Ela serve para acompanhar o progresso do treino — quantas atribuições mudaram de uma iteração para a seguinte:

In [ ]:
from scratch.linear_algebra import Vector

def num_differences(v1: Vector, v2: Vector) -> int:
    assert len(v1) == len(v2)
    return len([x1 for x1, x2 in zip(v1, v2) if x1 != x2])

assert num_differences([1, 2, 3], [2, 1, 3]) == 2
assert num_differences([1, 2], [1, 2]) == 0

Também precisamos de uma função que, dados os vetores e a atribuição de cada um a um grupo, calcule as médias dos grupos:

In [ ]:
from typing import List
import random
from scratch.linear_algebra import vector_mean

def cluster_means(k: int,
                  inputs: List[Vector],
                  assignments: List[int]) -> List[Vector]:
    # clusters[i] contém as entradas cuja atribuição é i
    clusters = [[] for i in range(k)]
    for input, assignment in zip(inputs, assignments):
        clusters[assignment].append(input)

    # se um grupo ficou vazio, use um ponto qualquer no lugar da média
    return [vector_mean(cluster) if cluster else random.choice(inputs)
            for cluster in clusters]

> **📌 Nota — Uma dívida do capítulo 4**
>
> A [seção 4.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) definiu `vector_mean` e prometeu que ela era "o coração do algoritmo de k-means". A linha acima é a cobrança dessa promessa: o passo de recalcular os centros é, literalmente, uma chamada a `vector_mean` por grupo.
>
> Repare também no `if cluster else random.choice(inputs)`. Um grupo pode ficar **vazio** — nada impede que, depois de uma rodada de atribuições, nenhum ponto seja o mais próximo de uma certa média. A média de uma coleção vazia não existe, então o código escolhe um ponto ao acaso para fazer as vezes de centro. É um remendo, não uma solução elegante, e ela tem consequência: é uma das razões pelas quais duas execuções do mesmo código sobre os mesmos dados podem terminar em lugares diferentes.

Com as duas peças, o algoritmo cabe numa classe. Não sabemos de antemão quantas iterações serão necessárias, então usamos `itertools.count`, que produz um iterável infinito, e saímos dele com um `return` quando as atribuições param de mudar:

In [ ]:
import itertools
import tqdm
from scratch.linear_algebra import squared_distance

class KMeans:
    def __init__(self, k: int) -> None:
        self.k = k                      # número de grupos
        self.means = None

    def classify(self, input: Vector) -> int:
        """devolve o índice do grupo mais próximo da entrada"""
        return min(range(self.k),
                   key=lambda i: squared_distance(input, self.means[i]))

    def train(self, inputs: List[Vector]) -> None:
        # Começa com atribuições aleatórias
        assignments = [random.randrange(self.k) for _ in inputs]

        with tqdm.tqdm(itertools.count()) as t:
            for _ in t:
                # Calcula as médias e encontra as novas atribuições
                self.means = cluster_means(self.k, inputs, assignments)
                new_assignments = [self.classify(input) for input in inputs]

                # Vê quantas atribuições mudaram, e se terminamos
                num_changed = num_differences(assignments, new_assignments)
                if num_changed == 0:
                    return

                # Senão, mantém as novas atribuições e recalcula as médias
                assignments = new_assignments
                self.means = cluster_means(self.k, inputs, assignments)
                t.set_description(f"changed: {num_changed} / {len(inputs)}")

A barra de progresso vem do `tqdm`, apresentado na [seção 7.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/07-um-parenteses-tqdm.html). Como o laço é sobre um gerador infinito, o `tqdm` não tem como estimar quanto falta — ele mostra só quantas iterações já passaram e a descrição que atualizamos a cada volta.

### Por que ele para, e por que isso não basta

Duas propriedades do algoritmo precisam ser ditas juntas, porque cada uma sozinha engana.

> **🔷 Conceito**
>
> **O k-means sempre converge.** O argumento é curto: existe um número finito de maneiras de atribuir *n* pontos a *k* grupos. Nenhum dos dois passos aumenta a soma das distâncias ao quadrado — recalcular a média de um grupo é, por definição, escolher o ponto que minimiza essa soma dentro dele; e reatribuir cada ponto ao centro mais próximo só pode diminuir a distância daquele ponto. Como o valor nunca sobe e há finitas configurações, o algoritmo não pode ficar rodando para sempre: em algum momento uma iteração não muda nada, e ele para.
>
> **Mas ele converge para um ótimo local.** "Não piorar" não é o mesmo que "chegar ao melhor". O ponto de parada é uma configuração da qual nenhum dos dois passos consegue sair — e qual delas você alcança depende inteiramente das atribuições aleatórias iniciais, sorteadas na primeira linha de `train`.

As duas afirmações juntas dão a forma exata do algoritmo: ele é confiável quanto a terminar e não é confiável quanto ao *onde*. Trocar a semente troca o resultado, e a [seção 17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html) vai mostrar duas execuções do mesmo código, sobre os mesmos dados e com o mesmo *k*, produzindo respostas visivelmente diferentes.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O equivalente do que você acabou de escrever:
>
> ```python
> from sklearn.cluster import KMeans
>
> modelo = KMeans(n_clusters=3, random_state=12).fit(X)
> modelo.cluster_centers_      # as médias
> modelo.labels_               # o grupo de cada ponto
> modelo.predict(X_novo)       # o nosso `classify`
> ```
>
> O que a biblioteca esconde é justamente o problema do ótimo local, e ela tem dois mecanismos contra ele:
>
> - **`init='k-means++'`** — este é o padrão, e é o que está agindo na chamada acima. Em vez de sortear as atribuições iniciais uniformemente, ele escolhe os centros iniciais um a um, com probabilidade proporcional ao **quadrado** da distância ao centro já escolhido mais próximo — o mesmo quadrado que aparece na soma que o `inertia_` reporta. Centros iniciais espalhados caem em ótimos locais melhores, e com muito mais frequência.
> - **`n_init`** — quantas vezes o algoritmo inteiro roda, cada uma com uma inicialização diferente, ficando com o melhor resultado. É força bruta contra o acaso: continua sendo um ótimo local, só que o melhor de *n*.
>
> **O segundo não está ligado por padrão, e é fácil supor que esteja.** O valor padrão de `n_init` é a string `'auto'`, e `'auto'` decide olhando para o `init`: com `'k-means++'` ele vale **1**, com `'random'` vale 10. Ou seja, na chamada padrão o algoritmo roda **uma única vez** — a biblioteca aposta que a inicialização esperta dispensa a repetição. Quem quiser o melhor de dez precisa pedir por escrito:
>
> ```python
> modelo = KMeans(n_clusters=3, n_init=10, random_state=12).fit(X)
> ```
>
> Nenhum dos dois **resolve** o problema; os dois o tornam menos provável. O atributo `inertia_` do modelo ajustado guarda exatamente a soma de distâncias ao quadrado que a [seção 17.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/04-escolhendo-k.html) vai calcular à mão — e é por essa soma que `n_init` escolhe o melhor, quando você o liga.

## Exemplo: Encontros

> **📌 Nota**
>
> Esta seção corresponde a *Example: Meetups*, do capítulo 20 de Grus (2019).

Para comemorar o crescimento da DataSciencester, a vice-presidente de Recompensas ao Usuário quer organizar encontros presenciais para os usuários da sua cidade, com cerveja, pizza e camisetas da empresa. Você conhece a localização de todos eles, e ela pede que você escolha locais de encontro que fiquem convenientes para todo mundo.

As localizações estão em quarteirões a leste e ao norte do centro da cidade — um par de números por usuário, vinte usuários:

In [ ]:
from typing import List
from scratch.linear_algebra import Vector

inputs: List[Vector] = [[-14, -5], [13, 13], [20, 23], [-19, -11], [-9, -16],
                        [21, 27], [-49, 15], [26, 13], [-46, 5], [-34, -1],
                        [11, 15], [-49, 0], [-22, -16], [19, 28], [-12, -8],
                        [-13, -19], [-41, 8], [-11, -6], [-25, -9], [-18, -3]]

len(inputs)

In [ ]:
# Figura: As localizações dos usuários da sua cidade
from matplotlib import pyplot as plt

xs, ys = zip(*inputs)

plt.scatter(xs, ys, color="#333388")
plt.title("Localizações dos usuários")
plt.xlabel("quarteirões a leste do centro")
plt.ylabel("quarteirões ao norte do centro")
plt.axis([-60, 40, -30, 40])
plt.gca().set_aspect('equal')     # eixos na mesma escala: distância é o assunto
plt.show()

Dependendo de como você olha, vê dois ou três grupos. É fácil enxergar porque os dados têm duas dimensões e cabem num gráfico; com mais dimensões, seria bem mais difícil de julgar a olho — e é exatamente por isso que se escreve o algoritmo.

### Três encontros

Suponha primeiro que o orçamento dê para três encontros:

In [ ]:
import random
from scratch.clustering import KMeans

random.seed(12)                    # para você obter o mesmo resultado
clusterer = KMeans(k=3)
clusterer.train(inputs)

for media in sorted(clusterer.means):
    print([round(coord, 2) for coord in media])

Três grupos, centrados em torno de [−43,8; 5,4], [−15,9; −10,3] e [18,3; 19,8]. É perto desses três pontos que se deve procurar um bar com espaço para o pessoal.

In [ ]:
# Figura: Os usuários divididos em três grupos. O número preto marca o centro de cada um.
import matplotlib.patheffects as pe
from scratch.linear_algebra import vector_mean

def desenha_grupos(grupos: List[List[Vector]], titulo: str) -> None:
    # A cor sai da posição no mapa, não do índice do grupo: o k-means numera
    # os grupos na ordem em que o sorteio os produziu, e essa ordem muda a
    # cada execução. Ordenando de leste para oeste, o grupo mais oriental
    # recebe sempre a primeira cor, o seguinte a segunda, e assim por diante
    # -- então as figuras deste capítulo ficam ancoradas umas nas outras.
    # (A âncora é o POSTO, não a região: com dois grupos em vez de três, o
    # oeste sobe de terceiro para segundo, e muda de cor junto.)
    grupos = sorted(grupos, key=lambda grupo: -vector_mean(grupo)[0])

    for i, (grupo, marca, cor) in enumerate(zip(grupos, ['D', 'o', '*'],
                                                ['r', 'g', 'b']), start=1):
        xs, ys = zip(*grupo)
        plt.scatter(xs, ys, color=cor, marker=marca)

        # e um número identificando o grupo, deslocado do centro. O centro cai
        # justamente onde os pontos são densos, então o algarismo é afastado
        # alguns pontos de TELA (não de dado, para o afastamento não mudar com
        # a escala) e ganha contorno branco, senão ele some em cima dos pontos
        # -- ou pior, os esconde: numa das figuras deste capítulo o número
        # cobria exatamente os dois pontos que o texto manda contar
        x, y = vector_mean(grupo)
        numero = plt.annotate(str(i), (x, y), textcoords='offset points',
                              xytext=(9, 9), fontsize=11, fontweight='bold')
        numero.set_path_effects([pe.withStroke(linewidth=3, foreground='white')])

    plt.title(titulo)
    plt.xlabel("quarteirões a leste do centro")
    plt.ylabel("quarteirões ao norte do centro")
    plt.axis([-60, 40, -30, 40])
    # a figura é de distância: um quarteirão no eixo x precisa medir o mesmo
    # que um quarteirão no eixo y, senão a geometria dos grupos sai deformada
    plt.gca().set_aspect('equal')
    plt.show()

atribuicoes_3 = [clusterer.classify(ponto) for ponto in inputs]
grupos_3 = [[p for p, a in zip(inputs, atribuicoes_3) if a == i] for i in range(3)]

desenha_grupos(grupos_3, "Localizações — 3 grupos")

> **⚠️ Atenção — O centro do grupo não é um usuário**
>
> Repare que os números pretos da figura caem em posições onde não há ponto nenhum. As médias são pontos do espaço, não elementos do conjunto de dados — e nada garante que exista um endereço utilizável ali.
>
> Achar um bar perto de [−15,9; −10,3] é trabalho de uma pessoa com um mapa. O algoritmo não sabe que existem ruas, nem que aquele quarteirão é um viaduto. Ele responde a uma pergunta geométrica; a pergunta original era logística, e a tradução de uma para a outra continua sendo humana.

### Dois encontros

Você mostra o resultado à vice-presidente, que informa que agora o orçamento só dá para **dois** encontros. Sem problema:

In [ ]:
random.seed(0)
clusterer_2 = KMeans(k=2)
clusterer_2.train(inputs)

for media in sorted(clusterer_2.means):
    print([round(coord, 2) for coord in media])

Um dos encontros continua perto de [18,3; 19,8]; o outro passa a ficar em [−25,9; −4,7].

In [ ]:
# Figura: Os mesmos usuários divididos em dois grupos
atribuicoes_2 = [clusterer_2.classify(ponto) for ponto in inputs]
grupos_2 = [[p for p, a in zip(inputs, atribuicoes_2) if a == i] for i in range(2)]

desenha_grupos(grupos_2, "Localizações — 2 grupos")

Comparando as duas figuras — a cor de cada grupo vem da posição dele no mapa, então elas são comparáveis a olho —, o grupo da direita ficou intacto, em vermelho nas duas, e os dois da esquerda viraram um só. Dá para conferir isso sem depender do olho, verificando se cada grupo da solução com *k* = 3 cabe inteiro dentro de algum grupo da solução com *k* = 2:

In [ ]:
def encaixa(grupos_finos: List[List[Vector]],
            grupos_grossos: List[List[Vector]]) -> bool:
    """todo grupo fino está contido em algum grupo grosso?"""
    grossos = [{tuple(p) for p in g} for g in grupos_grossos]
    return all(any({tuple(p) for p in fino} <= grosso for grosso in grossos)
               for fino in grupos_finos)

encaixa(grupos_3, grupos_2)

> **❗ Importante — Isso deu certo aqui, e não é uma garantia**
>
> É tentador ler a figura acima como "*k* = 2 é a versão grossa de *k* = 3", como se aumentar *k* fosse sempre subdividir os grupos que já existiam. **Não é.**
>
> O k-means com *k* = 2 e o k-means com *k* = 3 são duas otimizações independentes, cada uma com a sua inicialização aleatória e o seu ótimo local. Nada no algoritmo obriga a solução de *k* + 1 a refinar a de *k*, e a [seção 17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html) mostra um caso real em que mudar apenas a semente — nem sequer o *k* — reorganiza os grupos por inteiro.
>
> Neste conjunto de vinte pontos, com estas duas sementes, o encaixe aconteceu. É um fato sobre estes dados, não sobre o método. O algoritmo que **garante** essa propriedade por construção existe, e é o da [seção 17.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/06-clustering-hierarquico.html).

> **💡 Dica — Na prática: `scikit-learn`**
>
> O mesmo experimento, com a biblioteca:
>
> ```python
> from sklearn.cluster import KMeans
>
> modelo = KMeans(n_clusters=3, random_state=12).fit(inputs)
> modelo.cluster_centers_    # as três médias
> modelo.labels_             # o grupo de cada um dos 20 usuários
> modelo.inertia_            # a soma das distâncias ao quadrado
> ```
>
> Os centros não saem idênticos aos nossos, porque a inicialização é outra: o `scikit-learn` espalha os centros iniciais com `k-means++`, enquanto o nosso `train` sorteia as atribuições iniciais uniformemente. O callout da [seção 17.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/02-o-modelo.html) detalha esse mecanismo e o outro que a biblioteca oferece, o `n_init` — que, atenção, **não** vem ligado por padrão.
>
> Com vinte pontos em duas dimensões, essa diferença não aparece: os três grupos são separados o suficiente para que quase qualquer inicialização chegue lá. Ela começa a pesar quando os grupos se tocam, e vira o assunto principal quando os pontos são centenas de milhares — o caso da [seção 17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html).

## Escolhendo k

> **📌 Nota**
>
> Esta seção corresponde a *Choosing k*, do capítulo 20 de Grus (2019).

Na seção anterior, *k* foi escolhido por fatores completamente fora do nosso controle: primeiro o orçamento dava para três encontros, depois para dois. Em geral não é assim, e a pergunta cai no seu colo. Quantos grupos existem nestes dados?

Há várias formas de responder. Uma delas é razoavelmente fácil de entender: desenhar a soma dos erros ao quadrado — a distância de cada ponto à média do grupo dele — como função de *k*, e olhar onde o gráfico "dobra".

A função que calcula esse erro é curta:

In [ ]:
from typing import List
from scratch.linear_algebra import Vector, squared_distance
from scratch.clustering import KMeans

def squared_clustering_errors(inputs: List[Vector], k: int) -> float:
    """calcula o erro quadrático total de agrupar as entradas com k-means"""
    clusterer = KMeans(k)
    clusterer.train(inputs)
    means = clusterer.means
    assignments = [clusterer.classify(input) for input in inputs]

    return sum(squared_distance(input, means[cluster])
               for input, cluster in zip(inputs, assignments))

Aplicando aos vinte usuários da seção anterior, para todo *k* de 1 a 20:

In [ ]:
# Figura: Erro quadrático total em função do número de grupos
import random
from matplotlib import pyplot as plt

inputs: List[Vector] = [[-14, -5], [13, 13], [20, 23], [-19, -11], [-9, -16],
                        [21, 27], [-49, 15], [26, 13], [-46, 5], [-34, -1],
                        [11, 15], [-49, 0], [-22, -16], [19, 28], [-12, -8],
                        [-13, -19], [-41, 8], [-11, -6], [-25, -9], [-18, -3]]

random.seed(0)
ks = range(1, len(inputs) + 1)
errors = [squared_clustering_errors(inputs, k) for k in ks]

plt.plot(ks, errors, marker='.')
plt.xticks(list(ks))
plt.xlabel("k")
plt.ylabel("erro quadrático total")
plt.title("Erro total vs. nº de grupos")
plt.show()

O cotovelo está em *k* = 3, e é nítido. Os números por trás da curva deixam a leitura mais precisa:

In [ ]:
for k, erro in list(zip(ks, errors))[:8]:
    queda = "" if k == 1 else f"  ({100 * (erro / errors[k - 2] - 1):+.0f}%)"
    print(f"k = {k:2d}   erro = {erro:9.1f}{queda}")

De 1 para 2 grupos o erro cai 70%; de 2 para 3, outros 73%. De 3 para 4 ele cai 15%, e a partir dali — na escala do gráfico, que precisa acomodar os 15 mil do *k* = 1 — a curva vira quase uma reta rente ao eixo. (O tombo de 47% em *k* = 7, no meio dessa reta, não é uma segunda dobra escondida; ele tem a mesma explicação das *subidas* da curva, e mais adiante nesta seção voltamos a ele.) A leitura usual é essa: até *k* = 3 cada grupo novo está separando coisas que de fato estavam misturadas; depois disso, ele só está partindo grupos que já eram coesos. O método concorda com o que o olho já tinha dito na figura das localizações.

### A armadilha da métrica que sempre melhora

Só que essa concordância é sorte de um conjunto de dados fácil, e o método tem um problema que precisa ser dito com todas as letras.

> **❗ Importante — O erro só desce, então ele não pode escolher**
>
> Com o agrupamento **ótimo** para cada *k*, acrescentar um grupo nunca aumenta o erro total. O argumento é o mesmo de sempre: a melhor solução com *k* grupos continua disponível como candidata quando se permitem *k* + 1 — basta deixar o grupo extra com um ponto qualquer emprestado do vizinho —, então o melhor com *k* + 1 é, no máximo, tão ruim quanto o melhor com *k*.
>
> Leve isso ao extremo. Com *k* = *n*, cada ponto vira o próprio grupo, cada média coincide com o ponto que ela representa, e o erro é **exatamente zero**. Vinte grupos de um usuário cada: erro perfeito, e vinte festas com uma pessoa em cada.
>
> Uma métrica que atinge o valor perfeito na resposta mais inútil possível não pode, sozinha, escolher a resposta.

Esse defeito não é novo neste livro. Ele é, ponto por ponto, o mesmo do R² da [seção 12.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap12/05-qualidade-do-ajuste.html), que também nunca piora quando se acrescenta uma variável à regressão — inclusive uma coluna de puro ruído. Lá, uma métrica que nunca desce não servia para decidir quais variáveis merecem entrar no modelo; aqui, uma métrica que nunca sobe não serve para decidir quantos grupos existem. O remédio, nos dois casos, é o mesmo: a métrica interna informa, mas quem decide é um critério de fora dela.

E é aqui que a diferença entre este capítulo e os dezesseis anteriores fica concreta. Na regressão, o critério de fora existia e tinha nome: medir em dados que o modelo não viu. Aqui não há esse recurso, porque não há resposta certa nos dados novos tampouco. Sobram três caminhos, todos honestos e nenhum automático:

- **O uso decide.** Foi o que aconteceu na seção anterior: o orçamento dava para dois encontros, então *k* = 2. Uma gráfica que imprime cinco cores fixa *k* = 5. Não é uma limitação do método — é o método sendo usado do jeito certo, com a restrição real do problema no lugar de um critério inventado.
- **O gráfico decide, com o olho de quem lê.** É o cotovelo. Funciona quando existe um cotovelo, e boa parte dos conjuntos reais não tem nenhum: a curva desce suave, sem dobra, e a escolha vira arbitrária.
- **Uma métrica de separação decide.** Existem medidas que penalizam grupos ruins em vez de premiar grupos numerosos — a mais comum é a *silhueta*, que compara, para cada ponto, a distância média aos vizinhos do próprio grupo com a distância média ao grupo mais próximo. Diferente do erro quadrático, ela **não** melhora automaticamente com *k* maior, e por isso pode ter um máximo interior.

### A curva não desce sempre

Há um segundo detalhe na figura, e ele parece contradizer o que o parágrafo anterior afirmou. Olhe os valores de novo, agora até o fim:

In [ ]:
for k, erro in zip(ks, errors):
    seta = "  <-- subiu" if k > 1 and erro > errors[k - 2] else ""
    print(f"k = {k:2d}   erro = {erro:8.1f}{seta}")

> **⚠️ Atenção — Por que a curva medida sobe em alguns pontos**
>
> Cinco dos dezenove degraus da curva **sobem** em vez de descer. O erro em *k* = 9 é maior que em *k* = 8, e o de *k* = 18 é mais de seis vezes o de *k* = 17. Pior: em *k* = 20, com um grupo disponível por usuário, o erro deveria ser zero — e não é.
>
> Não há contradição com a caixa anterior. Aquele argumento é sobre o agrupamento **ótimo**, e o que a curva mostra é o resultado do **nosso algoritmo**, que não encontra o ótimo: cada ponto do gráfico é uma execução independente do k-means, com a sua própria atribuição inicial aleatória e o seu próprio ótimo local. Uma execução com *k* = 9 pode terminar num arranjo pior do que uma execução afortunada com *k* = 8.
>
> O mesmo mecanismo explica o degrau que chama atenção no outro sentido: o **−47% de *k* = 6 para *k* = 7**, no meio de uma faixa em que as quedas vinham na casa dos 10%. Não é que sete grupos revelem uma estrutura que seis não revelavam — é que a execução com *k* = 7 caiu num ótimo local afortunado, e a com *k* = 6, num ruim. Trocar a semente redistribui esses degraus, para cima e para baixo, sem mudar nada nos dados.
>
> O caso de *k* = 20 é o mais didático dos vinte. Existe uma solução de erro zero, ela é óbvia para qualquer pessoa, e o algoritmo não a encontra — porque começou com atribuições sorteadas, alguns grupos nasceram vazios e receberam pontos emprestados pelo `random.choice` de `cluster_means`, e a configuração resultante já era estável. Nenhum dos dois passos consegue escapar dela.
>
> Se você levar uma única coisa desta figura, que seja esta: **um número que sai de um algoritmo aleatório sem semente não é uma medida, é uma amostra de tamanho 1.** Rodar cada *k* dez vezes e ficar com o menor erro — que é o que o `scikit-learn` faz por padrão — deixaria a curva bem mais próxima da monótona. Não a tornaria monótona: só menos ruidosa.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O erro quadrático total que acabamos de calcular à mão é o atributo `inertia_` de um `KMeans` já ajustado, e a curva do cotovelo é um laço de três linhas:
>
> ```python
> from sklearn.cluster import KMeans
>
> erros = [KMeans(n_clusters=k, random_state=0).fit(inputs).inertia_
>          for k in range(1, 21)]
> ```
>
> Para a silhueta, `sklearn.metrics.silhouette_score(inputs, modelo.labels_)` devolve um número entre −1 e 1 — quanto mais alto, mais os grupos estão separados uns dos outros —, e o *k* escolhido é o que maximiza esse número. Repare na diferença de natureza: a inércia é minimizada e sempre pode ser reduzida; a silhueta é maximizada e tem um máximo interior. É por isso que ela serve de critério e a inércia, sozinha, não serve.
>
> Vale saber que existem também algoritmos que **não** exigem *k*: o `DBSCAN` do `scikit-learn` descobre o número de grupos a partir de um raio de vizinhança e de uma densidade mínima, e ainda marca pontos isolados como ruído em vez de forçá-los para dentro de algum grupo. Isso não faz a pergunta desaparecer — apenas a troca por outra, sobre o raio, que também não tem resposta correta.

## Exemplo: Clustering de Cores

> **📌 Nota**
>
> Esta seção corresponde a *Example: Clustering Colors*, do capítulo 20 de Grus (2019).

O vice-presidente de Brindes desenhou uns adesivos bonitos da DataSciencester para distribuir nos encontros. O problema é a gráfica: a impressora de adesivos aceita no máximo **cinco cores** por peça. E como a vice-presidente de Arte está de licença, ele pergunta se você não teria um jeito de modificar o desenho para que ele use só cinco cores.

Uma imagem é um arranjo bidimensional de pixels, e cada pixel é ele próprio um vetor de três dimensões — vermelho, verde e azul — que indica a cor. Fazer uma versão de cinco cores, então, é fazer duas coisas:

1. Escolher cinco cores.
2. Atribuir uma delas a cada pixel.

Isso é exatamente k-means com *k* = 5, em três dimensões: agrupar os pixels no espaço RGB e depois pintar cada pixel com a cor média do grupo dele. Repare que a estrutura espacial da imagem — quem está do lado de quem — é jogada fora: o algoritmo vê uma sacola de pontos coloridos, não um desenho.

### Carregando a imagem

O `matplotlib` lê arquivos de imagem através da biblioteca `pillow`:

In [ ]:
import matplotlib.image as mpimg

img = mpimg.imread("dados/imagem-cores.jpg") / 256   # reescala para [0, 1)

altura, largura, canais = img.shape
print(f"{largura} x {altura} pixels, {canais} canais por pixel")
print(f"total: {altura * largura:,} pixels".replace(",", "."))

> **❗ Importante — Por que este quadro, e não uma fotografia**
>
> A imagem é a *Composition II in Red, Blue, and Yellow* (1930), de Piet Mondrian. Não é uma escolha neutra, e vale dizer por quê: um quadro feito de campos chapados de cor primária é o caso mais favorável possível para o k-means. As cores da imagem já formam meia dúzia de amontoados apertados no espaço RGB, com quase nada entre eles — ou seja, os grupos existem *antes* do algoritmo, e a resposta é quase óbvia.
>
> Com uma fotografia comum, cheia de gradientes e sombras, não haveria amontoado nenhum: as cores preencheriam o espaço de forma contínua, o k-means dividiria esse contínuo em cinco fatias, e nenhum valor de *k* pareceria mais correto que os outros. Este exemplo é bonito porque é o caso fácil; a [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/04-escolhendo-k.html) é sobre o que fazer quando ele não é.

Cada linha de `img` é uma linha de pixels, e cada pixel é um vetor de três números entre 0 e 1:

In [ ]:
primeira_linha = img[0]
primeiro_pixel = primeira_linha[0]
vermelho, verde, azul = primeiro_pixel

[round(float(canal), 3) for canal in (vermelho, verde, azul)]

Por baixo, `img` é um array do `numpy` — é o que o `mpimg.imread` devolve. Nosso `KMeans` trabalha com listas de Python, então achatamos tudo numa lista única de pixels:

In [ ]:
# .tolist() converte um array do numpy numa lista de Python
pixels = [pixel.tolist() for row in img for pixel in row]

len(pixels), pixels[0]

### Cinco cores

São 355.200 pontos em três dimensões — quatro ordens de grandeza a mais que os vinte usuários da seção anterior. Este é um dos poucos treinos deste livro que demoram o suficiente para a barra de progresso do `tqdm` fazer diferença de verdade:

In [ ]:
import random
from scratch.clustering import KMeans

random.seed(0)

clusterer = KMeans(5)
clusterer.train(pixels)     # isso pode demorar

Numa máquina comum, esse `train` leva cerca de **16 segundos**. O valor exato depende do computador, e não é ele que importa: importa a ordem de grandeza — segundos, não milissegundos, para percorrer 355.200 listas de três elementos a cada iteração.

O treino devolve cinco cores. Vale olhar não só quais são, mas quanto da imagem cada uma cobre:

In [ ]:
from collections import Counter

atribuicoes = [clusterer.classify(pixel) for pixel in pixels]
tamanhos = Counter(atribuicoes)

for i, media in enumerate(clusterer.means):
    rgb = [round(255 * canal) for canal in media]
    fatia = 100 * tamanhos[i] / len(pixels)
    print(f"{str(rgb):18s} {tamanhos[i]:>7} pixels   ({fatia:4.1f}%)")

Um azul, um vermelho, um bege, um preto e um branco, escritos na escala de 0 a 255 que costuma aparecer em editores de imagem. O vermelho sozinho ocupa metade da tela. Recolorir a imagem é trocar cada pixel pela média do grupo dele:

In [ ]:
# Figura: À esquerda, o quadro original; à direita, a versão com as cinco cores escolhidas pelo k-means.
from typing import List
from scratch.linear_algebra import Vector
from matplotlib import pyplot as plt

def recolor(pixel: Vector) -> Vector:
    cluster = clusterer.classify(pixel)     # índice do grupo mais próximo
    return clusterer.means[cluster]         # média do grupo mais próximo

nova_img = [[recolor(pixel) for pixel in row]   # recolore esta linha de pixels
            for row in img]                     # para cada linha da imagem

fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
ax[0].imshow(img)
ax[0].axis('off')
ax[0].set_title("original")
ax[1].imshow(nova_img)
ax[1].axis('off')
ax[1].set_title("5 cores, semente 0")
plt.show()

À primeira vista funcionou: o vermelho, o branco, o preto e o azul saíram limpos, os campos ficaram chapados, e o resultado é imprimível. Agora olhe o canto inferior direito.

### O amarelo sumiu

O quadro tem *Yellow* no título, e o bloco amarelo não sobreviveu — ele foi pintado com o mesmo bege da borda da tela. Isso não é defeito de implementação; é o k-means fazendo exatamente o que se pediu.

In [ ]:
from scratch.linear_algebra import squared_distance

amarelo = img[545][560].tolist()     # um pixel dentro do bloco amarelo
print("cor do pixel: ", [round(255 * canal) for canal in amarelo])

grupo = clusterer.classify(amarelo)
print("caiu no grupo:", [round(255 * canal) for canal in clusterer.means[grupo]])

parecidos = sum(1 for p in pixels if squared_distance(p, amarelo) < 0.01)
print(f"pixels dessa cor: {parecidos} ({100 * parecidos / len(pixels):.2f}% da imagem)")

> **⚠️ Atenção — Uma cor rara é uma cor barata de sacrificar**
>
> Menos de 1% dos pixels são amarelos. O que o k-means minimiza é a soma dos erros ao quadrado sobre **todos** os pixels, e nessa conta os poucos milhares de pixels amarelos pesam menos que uma melhora minúscula espalhada pelos 177 mil pixels vermelhos. Gastar um dos cinco grupos no amarelo *piora* o objetivo.
>
> Ou seja: o algoritmo não errou. Ele otimizou a métrica que recebeu, e a métrica não sabe que "amarelo" é uma das três cores do título do quadro. Essa informação existe, mas é sua, não dos dados.
>
> Este é o mesmo aviso do [Capítulo 8](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/index.html) numa roupa nova. Lá, um modelo com boa acurácia podia ser inútil porque a classe que interessava era rara — e a acurácia, sendo uma média sobre todos os exemplos, não a enxergava. Aqui a classe rara é uma cor, e o efeito é idêntico: **uma média sobre o conjunto inteiro é cega para o que é raro, por mais importante que seja.**
>
> Se o amarelo importasse, as saídas seriam aumentar *k*, ponderar os pixels por alguma noção de importância, ou simplesmente fixar o amarelo à mão e deixar o k-means escolher as outras quatro cores. Nenhuma delas é o algoritmo se corrigindo; todas são uma pessoa recolocando a informação que a métrica não tinha.

### A mesma pergunta, outra resposta

A [seção 17.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/02-o-modelo.html) afirmou que o k-means converge sempre, mas para um ótimo local que depende da inicialização. Com vinte pontos bem separados, isso era teoria. Com 355.200 pixels, fica visível. Basta trocar a semente — mesmos dados, mesmo *k*, mesmo código:

In [ ]:
random.seed(1)
outro = KMeans(5)
outro.train(pixels)

for media in outro.means:
    print([round(255 * canal) for canal in media])

In [ ]:
# Figura: O mesmo k-means, com k = 5, sobre os mesmos 355.200 pixels — mudando apenas a semente do gerador aleatório.
outra_img = [[outro.means[outro.classify(pixel)] for pixel in row]
             for row in img]

plt.figure(figsize=(4.8, 4.8))
plt.imshow(outra_img)
plt.axis('off')
plt.title("5 cores, semente 1")
plt.show()

> **❗ Importante — O que aconteceu com esta execução**
>
> Compare as duas listas de cores. A segunda execução gastou **dois** dos cinco grupos em duas variantes do mesmo vermelho — `[217, 37, 25]` e `[224, 48, 43]`, uma diferença de poucas unidades por canal — e, para pagar por isso, fundiu o azul e o preto num único azul-marinho escuro.
>
> O efeito na figura é claro em dois lugares. O bloco azul do canto inferior esquerdo e as linhas pretas do quadro viraram a mesma cor: uma das três cores do título desapareceu, agora por outro motivo. E o campo vermelho, que na execução anterior era chapado, saiu **manchado** — as duas variantes não formam blocos limpos, e sim regiões irregulares que o olho não identifica como cores diferentes, só como sujeira na chapa. Numa peça impressa, isso seria um defeito de acabamento.
>
> Ou seja: dois dos cinco grupos, 40% do orçamento de cores da gráfica, foram gastos numa distinção que ninguém pediu e que ninguém enxerga como distinção.
>
> Nada disso é aleatório no sentido de "sem explicação". É um ótimo local: a partir daquelas atribuições iniciais, nenhum dos dois passos do algoritmo consegue mover um único ponto de forma a melhorar o erro. O algoritmo terminou corretamente, no lugar errado.
>
> A moral operacional é curta: **com k-means, uma execução não é um resultado.** Rode várias vezes, com inicializações diferentes, e compare — que é precisamente o que a opção `n_init` do `scikit-learn` faz por você.

> **💡 Dica — Na prática: `scikit-learn`**
>
> O mesmo trabalho, com a biblioteca:
>
> ```python
> from sklearn.cluster import KMeans
>
> modelo = KMeans(n_clusters=5, n_init=10, random_state=0).fit(pixels)
> nova = modelo.cluster_centers_[modelo.labels_].reshape(img.shape)
> ```
>
> Duas linhas, e uma fração dos dezesseis segundos que o nosso treino levou. A diferença de velocidade aqui é real e grande, porque o `scikit-learn` faz as contas em `numpy` compilado, sobre a matriz inteira de uma vez, enquanto o nosso código percorre 355.200 listas de três elementos em Python puro, uma a uma, a cada iteração. Esse é o preço da transparência que este livro escolheu pagar, e aqui ele aparece com unidade: dezesseis segundos, para uma conta que a biblioteca faz numa fração disso.
>
> Para imagens grandes existe ainda o `MiniBatchKMeans`, que a cada iteração atualiza as médias usando só uma amostra dos pontos — a mesma ideia do gradiente estocástico da [seção 5.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap05/06-minibatch-e-estocastico.html), aplicada a um algoritmo que não tem gradiente. Ele converge para um resultado um pouco pior e muito mais depressa.
>
> Fora do mundo do aprendizado de máquina, aliás, o problema tem nome próprio e soluções antigas: reduzir uma imagem a *n* cores se chama **quantização de cor**, e `PIL.Image.quantize(colors=5)` faz isso com algoritmos desenhados especificamente para o caso — mediana de cortes, octree — que costumam preservar melhor as cores raras, justamente porque não minimizam uma média sobre todos os pixels.

## Clustering Hierárquico Bottom-Up

> **📌 Nota**
>
> Esta seção corresponde a *Bottom-Up Hierarchical Clustering*, do capítulo 20 de Grus (2019).

O k-means começa pela resposta — *k* grupos — e vai ajustando até que ela pare de mudar. Uma abordagem alternativa faz o contrário: começa com o máximo de grupos possível e vai juntando.

> **🔷 Conceito**
>
> O clustering hierárquico **bottom-up** é este algoritmo:
>
> 1. Faça de cada entrada um grupo de um elemento só.
> 2. Enquanto houver mais de um grupo, encontre os dois grupos mais próximos e funda-os.
>
> No fim sobra um grupo gigante contendo tudo. Se, no caminho, você anotar a **ordem** em que as fusões aconteceram, pode desfazê-las: para obter três grupos, basta desfazer as duas últimas fusões.

Repare no que isso muda. O k-means devolve uma partição para o *k* que você pediu, e nada mais. O bottom-up devolve uma **hierarquia** — uma árvore de fusões — da qual se extrai qualquer número de grupos depois, sem treinar de novo.

### Representando os grupos

Os valores vivem em grupos-**folha**, representados como `NamedTuple`:

In [ ]:
from typing import NamedTuple, Union, List
from scratch.linear_algebra import Vector

class Leaf(NamedTuple):
    value: Vector

leaf1 = Leaf([10,  20])
leaf2 = Leaf([30, -15])

E as fusões produzem grupos **merged**, também `NamedTuple`, que guardam os filhos e a ordem em que a fusão aconteceu:

In [ ]:
class Merged(NamedTuple):
    children: tuple
    order: int

merged = Merged((leaf1, leaf2), order=1)

Cluster = Union[Leaf, Merged]

> **📌 Nota**
>
> `Merged.children` deveria ser anotado como `Tuple[Cluster, Cluster]`, mas `Cluster` é um tipo **recursivo** — ele aparece dentro da própria definição —, e o verificador `mypy` não aceita isso. Fica `tuple`, uma anotação mais fraca do que a realidade. Vale saber que acontece: nem sempre o sistema de tipos consegue escrever o que o código de fato faz.

Um grupo pode conter outros grupos, então recuperar os valores dentro dele é uma recursão:

In [ ]:
def get_values(cluster: Cluster) -> List[Vector]:
    if isinstance(cluster, Leaf):
        return [cluster.value]
    else:
        return [value
                for child in cluster.children
                for value in get_values(child)]

assert get_values(merged) == [[10, 20], [30, -15]]

### Distância entre grupos, e a decisão escondida nela

Para fundir "os dois grupos mais próximos" é preciso definir o que é a distância entre dois *grupos*. Aqui aparece a decisão mais importante da seção, e ela não tem resposta única.

In [ ]:
from typing import Callable
from scratch.linear_algebra import distance

def cluster_distance(cluster1: Cluster,
                     cluster2: Cluster,
                     distance_agg: Callable = min) -> float:
    """
    calcula todas as distâncias par a par entre cluster1 e cluster2
    e aplica a função de agregação distance_agg à lista resultante
    """
    return distance_agg([distance(v1, v2)
                         for v1 in get_values(cluster1)
                         for v2 in get_values(cluster2)])

A função calcula **todas** as distâncias entre um ponto de um grupo e um ponto do outro, e depois resume essa lista com uma função de agregação. Qual função você passa é o que se chama, na literatura, de **critério de ligação**:

- `min` — a distância entre os grupos é a do par mais próximo. Dois grupos se fundem assim que *se tocam* em qualquer ponto, o que produz grupos alongados, em forma de cadeia.
- `max` — a distância é a do par mais distante. Fundem-se os dois grupos que cabem na menor bola possível, o que produz grupos compactos e arredondados.
- A média das distâncias também é comum, e fica entre as duas.

O padrão da função é `min`. Guarde isso: o mesmo algoritmo, sobre os mesmos dados, vai dar respostas diferentes conforme esse argumento, e nenhuma delas é a errada.

### O algoritmo

Falta anotar a ordem das fusões. A convenção é que números **menores** representam fusões **mais tardias** — a última fusão de todas recebe ordem 0. Assim, "desfazer as fusões mais recentes" é percorrer as ordens da menor para a maior. Folhas nunca foram fundidas, então recebem infinito:

In [ ]:
def get_merge_order(cluster: Cluster) -> float:
    if isinstance(cluster, Leaf):
        return float('inf')  # nunca foi fundido
    else:
        return cluster.order

def get_children(cluster: Cluster):
    if isinstance(cluster, Leaf):
        raise TypeError("Leaf has no children")
    else:
        return cluster.children

Com isso, o algoritmo inteiro:

In [ ]:
from typing import Tuple

def bottom_up_cluster(inputs: List[Vector],
                      distance_agg: Callable = min) -> Cluster:
    # Começa com todas as folhas
    clusters: List[Cluster] = [Leaf(input) for input in inputs]

    def pair_distance(pair: Tuple[Cluster, Cluster]) -> float:
        return cluster_distance(pair[0], pair[1], distance_agg)

    # enquanto sobrar mais de um grupo...
    while len(clusters) > 1:
        # encontra os dois grupos mais próximos
        c1, c2 = min(((cluster1, cluster2)
                      for i, cluster1 in enumerate(clusters)
                      for cluster2 in clusters[:i]),
                      key=pair_distance)

        # remove os dois da lista de grupos
        clusters = [c for c in clusters if c != c1 and c != c2]

        # funde os dois, com ordem de fusão = nº de grupos restantes
        merged_cluster = Merged((c1, c2), order=len(clusters))

        # e acrescenta a fusão
        clusters.append(merged_cluster)

    # quando sobra um só grupo, devolve
    return clusters[0]

E a função que extrai o número de grupos que se quiser, desfazendo fusões:

In [ ]:
def generate_clusters(base_cluster: Cluster,
                      num_clusters: int) -> List[Cluster]:
    # começa com uma lista contendo só o grupo-base
    clusters = [base_cluster]

    # enquanto não houver grupos suficientes...
    while len(clusters) < num_clusters:
        # escolhe o último grupo a ter sido fundido
        next_cluster = min(clusters, key=get_merge_order)
        # remove-o da lista
        clusters = [c for c in clusters if c != next_cluster]

        # e acrescenta os filhos dele à lista (ou seja, desfaz a fusão)
        clusters.extend(get_children(next_cluster))

    # quando houver grupos suficientes...
    return clusters

### Os mesmos vinte usuários

Voltemos às localizações da [seção 17.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/03-exemplo-encontros.html):

In [ ]:
inputs: List[Vector] = [[-14, -5], [13, 13], [20, 23], [-19, -11], [-9, -16],
                        [21, 27], [-49, 15], [26, 13], [-46, 5], [-34, -1],
                        [11, 15], [-49, 0], [-22, -16], [19, 28], [-12, -8],
                        [-13, -19], [-41, 8], [-11, -6], [-25, -9], [-18, -3]]

base_cluster = bottom_up_cluster(inputs)
tres_grupos = [get_values(cluster)
               for cluster in generate_clusters(base_cluster, 3)]

sorted(len(grupo) for grupo in tres_grupos)

In [ ]:
# Figura: Três grupos bottom-up com ligação mínima. O grupo das estrelas azuis tem catorze dos vinte usuários.
from matplotlib import pyplot as plt
import matplotlib.patheffects as pe
from scratch.linear_algebra import vector_mean

def desenha_grupos(grupos: List[List[Vector]], titulo: str) -> None:
    # mesma função da seção 17.3: a cor vem do posto do grupo na ordem
    # leste-para-oeste, não do índice que o algoritmo lhe deu -- é o que
    # torna as figuras dos dois algoritmos comparáveis entre si
    grupos = sorted(grupos, key=lambda grupo: -vector_mean(grupo)[0])

    for i, (grupo, marca, cor) in enumerate(zip(grupos, ['D', 'o', '*'],
                                                ['r', 'g', 'b']), start=1):
        xs, ys = zip(*grupo)
        plt.scatter(xs, ys, color=cor, marker=marca)

        x, y = vector_mean(grupo)
        numero = plt.annotate(str(i), (x, y), textcoords='offset points',
                              xytext=(9, 9), fontsize=11, fontweight='bold')
        numero.set_path_effects([pe.withStroke(linewidth=3, foreground='white')])

    plt.title(titulo)
    plt.xlabel("quarteirões a leste do centro")
    plt.ylabel("quarteirões ao norte do centro")
    plt.axis([-60, 40, -30, 40])
    plt.gca().set_aspect('equal')
    plt.show()

desenha_grupos(tres_grupos, "3 grupos bottom-up, ligação mínima")

O resultado é bem diferente do k-means. Catorze dos vinte usuários caíram num grupo só — as estrelas azuis, que vão do extremo oeste do mapa até o centro-sul —, enquanto os seis do canto nordeste, que o k-means tratava como um grupo coeso, foram partidos em quatro, os losangos vermelhos, mais dois, os círculos verdes.

É o efeito da ligação `min` que a seção anterior antecipou: como basta que dois grupos *se toquem* em um ponto para se fundirem, os grupos crescem em cadeia. Cada usuário do meio do mapa serve de ponte entre o vizinho da esquerda e o da direita, e a cadeia inteira vira um grupo.

### Trocando um argumento

Agora com `max`, e nada mais:

In [ ]:
# Figura: Três grupos bottom-up com ligação máxima — sobre exatamente os mesmos vinte pontos.
base_cluster_max = bottom_up_cluster(inputs, max)
tres_grupos_max = [get_values(cluster)
                   for cluster in generate_clusters(base_cluster_max, 3)]

desenha_grupos(tres_grupos_max, "3 grupos bottom-up, ligação máxima")

Esta é a mesma divisão que o k-means encontrou com *k* = 3. Não "parecida": a mesma — e, como a cor de cada grupo vem da posição dele no mapa, as duas figuras podem ser sobrepostas. Dá para verificar sem depender do olho, comparando as duas partições ponto a ponto:

In [ ]:
import random
from scratch.clustering import KMeans

random.seed(12)
clusterer = KMeans(k=3)
clusterer.train(inputs)

atribuicoes = [clusterer.classify(ponto) for ponto in inputs]
grupos_kmeans = [[p for p, a in zip(inputs, atribuicoes) if a == i]
                 for i in range(3)]

def como_conjunto(grupos):
    """a partição, sem depender da ordem dos grupos nem dos pontos"""
    return sorted(sorted(tuple(ponto) for ponto in grupo) for grupo in grupos)

assert como_conjunto(grupos_kmeans) == como_conjunto(tres_grupos_max)

for grupo in tres_grupos_max:
    print(f"{len(grupo):2d} usuários, centro em "
          f"{[round(coord, 2) for coord in vector_mean(grupo)]}")

> **🔷 Conceito**
>
> Dois algoritmos sem nenhum parentesco — um que sorteia atribuições e itera até convergir, outro que funde pares deterministicamente de baixo para cima — chegaram exatamente à mesma resposta. E o mesmo algoritmo hierárquico, com um único argumento trocado, chegou a uma resposta completamente diferente.
>
> A lição não é "a ligação máxima é melhor". É que **agrupar não é uma operação, é uma família de decisões**: qual algoritmo, qual noção de distância entre grupos, quantos grupos. Cada combinação define o que conta como "parecido", e o conjunto de dados aceita todas elas sem reclamar.
>
> Quando duas escolhas independentes concordam — como acontece aqui —, isso é uma evidência real de que a estrutura está nos dados e não no método. É o mais próximo de uma validação que este capítulo consegue oferecer, e repare no quanto é mais fraco do que uma acurácia medida em dados de teste.

### Três diferenças práticas em relação ao k-means

**Não há semente.** Repare que nenhum agrupamento desta seção precisou de `random.seed` — o único que fixa semente é o treino do k-means, para comparação. O bottom-up é determinístico: dados os mesmos pontos e a mesma ligação, ele sempre devolve a mesma árvore. Depois de duas seções lidando com ótimos locais e execuções que discordam entre si, essa propriedade não é pouca coisa.

**A hierarquia é aninhada por construção.** A [seção 17.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/03-exemplo-encontros.html) avisou que a solução do k-means com *k* + 1 grupos não é necessariamente um refinamento da solução com *k*. Aqui é, e não por sorte: os três grupos saem de desfazer uma fusão a mais na mesma árvore que produziu os dois. É a mesma estrutura, cortada em outra altura.

**E ele é caro.** Muito caro. Dá para contar quantas distâncias ponto a ponto o algoritmo calcula, aproveitando que `distance_agg` recebe a lista inteira:

In [ ]:
contagem = 0

def min_contando(distancias: List[float]) -> float:
    global contagem
    contagem += len(distancias)
    return min(distancias)

bottom_up_cluster(inputs, min_contando)
print(f"distâncias ponto a ponto calculadas: {contagem}")

Três mil distâncias para agrupar vinte pontos. Compare com o k-means: cada passagem dele compara os vinte pontos com as três médias, o que dá sessenta distâncias, e ele converge em pouquíssimas passagens — o total fica na casa da centena. A razão da diferença é que `cluster_distance` recalcula, a cada fusão, todas as distâncias entre todos os pares de pontos dos dois grupos — inclusive as que já haviam sido calculadas dezenas de vezes antes. É uma implementação assombrosamente ineficiente, e a correção é conhecida: pré-calcular a distância entre cada par de entradas uma única vez e consultar essa tabela dentro de `cluster_distance`. Uma versão realmente eficiente ainda guardaria as distâncias entre grupos de um passo para o seguinte.

> **⚠️ Atenção — Um defeito silencioso no `bottom_up_cluster`**
>
> O `bottom_up_cluster` remove os dois grupos fundidos com `[c for c in clusters if c != c1 and c != c2]`, e `c != c1` compara `NamedTuple`s **por valor**. Duas folhas com as mesmas coordenadas são objetos distintos — `is` diria que não são a mesma —, mas são **iguais** por `==`, e é a igualdade que o filtro usa. O defeito mora exatamente nessa distinção: o filtro pretendia remover *aqueles dois* grupos e remove *todos os que se parecem com eles*.
>
> Enquanto houver no máximo duas entradas idênticas, isso não aparece: as duas são exatamente o par mais próximo (distância zero), são fundidas juntas, e a remoção dupla não faz falta. Com três ou mais, os pontos excedentes são apagados da lista sem nunca entrarem na árvore:

In [ ]:
for extras in [1, 2, 3, 4]:
    entradas = inputs + [[13, 13]] * extras
    arvore = bottom_up_cluster(entradas)
    print(f"{len(entradas)} entradas -> {len(get_values(arvore))} folhas na árvore")

> Nenhuma exceção, nenhum aviso: os pontos simplesmente somem. E coordenadas repetidas são comuns em dados reais — duas medições idênticas, dois usuários no mesmo prédio, dois pixels da mesma cor.
>
> Um livro inteiro depois, o padrão deve estar reconhecível: **o erro perigoso não é o que estoura, é o que devolve uma resposta plausível.** Este devolve uma árvore perfeitamente bem formada, com menos dados do que você entregou.

> **💡 Dica — Na prática: `scikit-learn` e `scipy`**
>
> O equivalente na biblioteca é o `AgglomerativeClustering`, e o critério de ligação é um parâmetro explícito:
>
> ```python
> from sklearn.cluster import AgglomerativeClustering
>
> AgglomerativeClustering(n_clusters=3, linkage='single').fit(inputs)    # o nosso min
> AgglomerativeClustering(n_clusters=3, linkage='complete').fit(inputs)  # o nosso max
> AgglomerativeClustering(n_clusters=3, linkage='average').fit(inputs)   # a média
> AgglomerativeClustering(n_clusters=3, linkage='ward').fit(inputs)      # outro critério
> ```
>
> `ward` é o padrão da biblioteca e não tem equivalente no que escrevemos: em vez de agregar distâncias entre pontos, ele funde o par de grupos que menos aumenta a soma dos erros ao quadrado — a mesma quantidade que o k-means minimiza. É o critério que costuma dar resultados mais parecidos com os do k-means, o que é coerente com o que a ligação `max` mostrou acima.
>
> Para desenhar a árvore de fusões — o **dendrograma**, que é a visualização natural deste algoritmo e que não chegamos a produzir aqui —, a ferramenta é o `scipy`:
>
> ```python
> from scipy.cluster.hierarchy import linkage, dendrogram
>
> Z = linkage(inputs, method='single')
> dendrogram(Z)
> ```
>
> O `Z` que o `scipy` devolve é a mesma informação que a nossa árvore de `Merged` guarda: quem se fundiu com quem, em que ordem e a que distância. A diferença é que ele registra também a *altura* de cada fusão, e é essa altura que permite cortar a árvore por distância — "todos os grupos cujos membros estão a menos de 10 quarteirões uns dos outros" — em vez de por número de grupos. É outra forma de não ter que escolher *k*.

### Fim do livro

Este é o último algoritmo do livro, e é apropriado que ele venha do único capítulo sem resposta certa.

Os dezesseis capítulos anteriores construíram uma sequência confortável: um modelo, uma métrica, um número que diz se funcionou. A regressão tinha o R², o classificador tinha a acurácia, a rede neural tinha a perda no conjunto de teste. Havia sempre um lugar para onde olhar e saber se o trabalho estava certo. Este capítulo tirou isso, e o que sobrou — julgar se um agrupamento é útil olhando para o que caiu dentro dele — é, na verdade, o estado normal de boa parte do trabalho real. A confortável era a exceção.

O [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html) prometeu duas coisas. A primeira era abrir as caixas-pretas, e vale conferir o que foi cumprido. Você escreveu, do zero e em Python puro, os vetores e o produto escalar, o gradiente descendente que treina cinco dos modelos, o classificador por vizinhos, o Naive Bayes, três formas de regressão, as árvores de decisão, a retropropagação de uma rede neural, e agora dois algoritmos de agrupamento. Nenhum deles é mais uma chamada de função que devolve um número: são coisas que você viu por dentro e pode julgar.

A segunda promessa era reconhecer usos da área que ajudam pessoas e usos que as manipulam — **e perceber que a técnica é a mesma nos dois**. Este capítulo é onde ela se paga, e não por acaso: o k-means que agrupou vinte usuários por localização, para decidir onde fazer encontros, é o mesmo que segmenta um cadastro de eleitores por idade, renda e composição familiar, para decidir que mensagem cada grupo recebe. O capítulo 1 citou as duas campanhas presidenciais americanas quando você ainda não sabia escrever nem uma coisa nem outra. Agora sabe, e a distância entre os dois usos não está em lugar nenhum do algoritmo — está inteira em quem escolheu os atributos, em quem definiu o que é um grupo útil, e no que se faz com os grupos depois. O código é indiferente. Você não precisa ser.

Três dívidas ficaram, e é melhor nomeá-las do que fingir que não existem. A primeira é de ferramenta: o código deste livro é lento e verboso de propósito, e nada do que você escreveu aqui deve ir para produção — em produção usa-se `numpy`, `pandas` e `scikit-learn`, e agora você tem como avaliar o que essas bibliotecas fazem em vez de apenas confiar. A segunda é de escala: os conjuntos de dados aqui têm dezenas ou centenas de linhas, e problemas de verdade trazem dificuldades — dados sujos, faltantes, enviesados — que nenhum capítulo resolveu. A terceira é de assunto: processamento de linguagem natural, análise de redes, sistemas de recomendação e o resto do que o livro-texto cobre em seus capítulos finais ficaram fora do escopo desta disciplina, e o livro-texto continua ali para quem quiser seguir.

Fica também o hábito que este material tentou construir, e que é o mais transferível de tudo: **desconfiar de um resultado que parece bom.** A média que não significava nada no [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html), o classificador que lia os cortes dos próprios dados, o R² que subia com colunas de ruído, a curva de erro que descia sozinha, o amarelo que sumiu de um quadro cujo título tem *Yellow*, a árvore de fusões que devolveu menos pontos do que recebeu. Nenhum desses erros estourou; todos devolveram uma resposta plausível. A diferença entre quem os pega e quem não os pega não está na ferramenta — está em saber o que a ferramenta está fazendo.

É para isso que se abrem caixas-pretas.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 20 de Grus (2019) faz duas sugestões.

A primeira é o [módulo `sklearn.cluster`](https://scikit-learn.org/stable/modules/clustering.html) do `scikit-learn`, que reúne uma dúzia de algoritmos de agrupamento — entre eles o `KMeans` e o `AgglomerativeClustering` com o critério de ligação de **Ward**, que funde o par de grupos que menos aumenta a soma dos erros ao quadrado. A página traz uma tabela comparando os algoritmos por escalabilidade, parâmetros exigidos e o formato de grupo que cada um consegue encontrar; é uma boa forma de ver, de uma vez, quantas definições diferentes de "grupo" convivem na área.

A segunda é o [`SciPy`](https://docs.scipy.org/doc/scipy/reference/cluster.html), que tem dois módulos de agrupamento: `scipy.cluster.vq`, com k-means (o `vq` é de *vector quantization*, exatamente o uso da [seção 17.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/05-exemplo-clustering-de-cores.html)), e `scipy.cluster.hierarchy`, com uma variedade de algoritmos hierárquicos e com o `dendrogram`, que desenha a árvore de fusões que a [seção 17.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap17/06-clustering-hierarquico.html) constrói mas não chega a visualizar.

Para o tratamento estatístico de métodos não supervisionados — k-means, clustering hierárquico e também PCA —, James et al. (2021) dedica um capítulo introdutório e Hastie et al. (2009) um tratamento mais aprofundado, incluindo a *estatística de gap*, um critério para escolher *k* mais defensável que o cotovelo.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed.. Springer. 2009.
- **James; Witten; Hastie; Tibshirani**. *An Introduction to Statistical Learning*. 2nd ed.. Springer. 2021.